### RAG Pipeline

In [2]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 

/Users/prateet/Documents/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Data Ingestion 
### 1) Read all the pdfs files
def process_all_pdfs(pdf_directory):
    all_pdf_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try: 
            loader = PyMuPDFLoader(str(pdf_file))
            doc = loader.load()

            for d in doc: 
                d.metadata["source_file"] = pdf_file.name 
                d.metadata["file_type"] = 'pdf'

            all_pdf_docs.extend(doc)

            print(f"Loaded {len(doc)} Pages")
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal documents loaded: {len(all_pdf_docs)}")
    return all_pdf_docs

all_docs = process_all_pdfs("../data")
all_docs 


Found 3 PDF files to process

Processing CSCI_530_Research_Paper.pdf
Loaded 9 Pages

Processing pre_midterm_slides.pdf
Loaded 208 Pages

Processing SmartMedAI_Research_Paper.pdf
Loaded 6 Pages

Total documents loaded: 223


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-12T06:09:59+00:00', 'source': '../data/pdf/CSCI_530_Research_Paper.pdf', 'file_path': '../data/pdf/CSCI_530_Research_Paper.pdf', 'total_pages': 9, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-12T06:09:59+00:00', 'trapped': '', 'modDate': 'D:20251212060959Z', 'creationDate': 'D:20251212060959Z', 'page': 0, 'source_file': 'CSCI_530_Research_Paper.pdf', 'file_type': 'pdf'}, page_content='CryptoFL: A Novel Architecture For Securing Federated\nLearning with Cryptographic Techniques\nName: Prateet Mishra\nUSCID: 9074068892\nCourse: CSCI-530 Security Systems\nI have read the Guide to Avoiding Plagiarism published by the student affairs office. I\nunderstand what is expected of me with respect to properly citing sources, and how to avoid\nrepresenting the work of others as my own. The material in this paper was written by me,\nexcept

In [4]:
### 2) Chunking of files
def split_document(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len, 
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample Chunk")
        print(f"Content {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_document(all_docs)
chunks

Split 223 documents into 291 chunks

Example Chunk
Content CryptoFL: A Novel Architecture For Securing Federated
Learning with Cryptographic Techniques
Name: Prateet Mishra
USCID: 9074068892
Course: CSCI-530 Security Systems
I have read the Guide to Avoiding ...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-12T06:09:59+00:00', 'source': '../data/pdf/CSCI_530_Research_Paper.pdf', 'file_path': '../data/pdf/CSCI_530_Research_Paper.pdf', 'total_pages': 9, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-12T06:09:59+00:00', 'trapped': '', 'modDate': 'D:20251212060959Z', 'creationDate': 'D:20251212060959Z', 'page': 0, 'source_file': 'CSCI_530_Research_Paper.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-12T06:09:59+00:00', 'source': '../data/pdf/CSCI_530_Research_Paper.pdf', 'file_path': '../data/pdf/CSCI_530_Research_Paper.pdf', 'total_pages': 9, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-12T06:09:59+00:00', 'trapped': '', 'modDate': 'D:20251212060959Z', 'creationDate': 'D:20251212060959Z', 'page': 0, 'source_file': 'CSCI_530_Research_Paper.pdf', 'file_type': 'pdf'}, page_content='CryptoFL: A Novel Architecture For Securing Federated\nLearning with Cryptographic Techniques\nName: Prateet Mishra\nUSCID: 9074068892\nCourse: CSCI-530 Security Systems\nI have read the Guide to Avoiding Plagiarism published by the student affairs office. I\nunderstand what is expected of me with respect to properly citing sources, and how to avoid\nrepresenting the work of others as my own. The material in this paper was written by me,\nexcept

### Embedding and VectorDB 

In [6]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Dict, Any 
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager: 
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None 
        self._load_model()
    
    def _load_model(self):
        try: 
            print(f"Loading model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully Loaded {self.model_name}. Embedding Dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e: 
            print(f"Error Loading Model {e}")
            raise
    
    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        if not self.model: 
            raise ValueError("Model not loaded!")
        
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading model sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2594.27it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully Loaded sentence-transformers/all-MiniLM-L6-v2. Embedding Dimensions: 384


In [8]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json",
)
print("Hub download works")

Hub download works


### Vector Store

In [9]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_docs", persistent_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None 
        self.collection = None 
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persistent_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection {self.collection_name}")
            print(f"Existing documents in the collection {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection pdf_docs
Existing documents in the collection 291


In [10]:
all_texts = [doc.page_content for doc in chunks]
all_texts

['CryptoFL: A Novel Architecture For Securing Federated\nLearning with Cryptographic Techniques\nName: Prateet Mishra\nUSCID: 9074068892\nCourse: CSCI-530 Security Systems\nI have read the Guide to Avoiding Plagiarism published by the student affairs office. I\nunderstand what is expected of me with respect to properly citing sources, and how to avoid\nrepresenting the work of others as my own. The material in this paper was written by me,\nexcept for such material that is quoted or indented and properly cited to indicated the sources\nof the material. I understand that using the words of others, and simply tagging the sentence,\nparagraph, or section with a tag to the copied source does not constitute proper citation and\nthat if such materiel is used verbatim or paraphrased it must be specifically conveyed (such as\nthrough the use of quotation marks or indentation) together with the citation. I further\nunderstand that overuse of properly cited quotations to avoid conveying the info

In [11]:
embeddings = embedding_manager.generate_embeddings(all_texts)
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 291 texts


Batches: 100%|██████████| 10/10 [00:01<00:00,  9.68it/s]


Generated embeddings with shape (291, 384)
Adding 291 documents to vector store...
Successfully added 291 documents to vector store
Total documents in collection: 582


### Retriever Pipeline from Vector Store

In [12]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"retrieving documents for query {query}")
        print(f"Top k: {top_k} and Score Threshold: {score_threshold}")

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=Retriever(vectorstore,embedding_manager)

In [13]:
rag_retriever.retrieve("Security and privacy in federated learning.")

retrieving documents for query Security and privacy in federated learning.
Top k: 5 and Score Threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.66it/s]

Generated embeddings with shape (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4a3f81a0_9',
  'content': 'ated challenges and "trade-off" conditions.\nII.\nRELATED WORK\nA. Security and Privacy in Federated Learning\nMany researchers have published studies exploring\nthe risks associated with federated model to train\nmachine learning algorithms from the point of view\nof confidentiality and integrity [5], [9]. Jimenez\nGutierrez et al. provide a high-level overview of\nboth threats to confidentiality and integrity posed\nby federated learning, as well as specific exam-\nples of possible attack methods and defenses [5].\nAlthough these studies provide very valuable in-\nformation regarding threats to confidentiality and\nintegrity in federated learning, they do not main-\ntain privacy and robustness when using federated\nlearning without further measures to secure data.\nAccording to Zhang et al., trustworthy federated\nlearning is characterized as an integrated frame-\nwork in which privacy, robustness, and accountabil-\nity must be addressed jointl

### Integrating VectorDB Context Pipeline with LLM

In [14]:
### Simple Rag Pipeline with Groq
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ-API-KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant",temperature=0.1, max_tokens=1024)

### now we are making a simple rag function with both context retrieval and output generation

def simple_rag(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    prompt = f""" Use the following context to answer the quetions carefully. 

    Context: {context}
    Question: {query}
    Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content 

In [15]:
answer = simple_rag("What is UltraMedical Dataset?",rag_retriever,llm)
print(answer)

retrieving documents for query What is UltraMedical Dataset?
Top k: 3 and Score Threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 162.44it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


The UltraMedical dataset is a vast and meticulously selected set of biomedical instructions used to optimize language models. It contains over 400,000 instructions, with over 100,000 of those instructions annotated with model-generated preferences. The addition of human-reviewed preference examples makes it valuable for developing dependable and specialized biomedical AI systems.


In [16]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("What is federated learning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

retrieving documents for query What is federated learning?
Top k: 3 and Score Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.00it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)


Answer: Federated learning is a privacy-preserving method that allows multiple clients to jointly train a global model without sharing their local data. Instead, clients share model updates (gradients and parameters) with a central server, which aggregates the updates to form a global model. This process is repeated over several rounds.
Sources: [{'source': 'CSCI_530_Research_Paper.pdf', 'page': 2, 'score': 0.4511500597000122, 'preview': 'ated challenges and "trade-off" conditions.\nII.\nRELATED WORK\nA. Security and Privacy in Federated Learning\nMany researchers have published studies exploring\nthe risks associated with federated model to train\nmachine learning algorithms from the point of view\nof confidentiality and integrity [5], [9]...'}, {'source': 'CSCI_530_Research_Paper.pdf', 'page': 2, 'score': 0.4511500597000122, 'preview': 'ated challenges and "trade-off" conditions.\nII.\nRELATED WORK\nA. Security and Privacy in Federated Learning\nMany researchers have published studie

In [17]:
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("name the three aspects of security.", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

retrieving documents for query name the three aspects of security.
Top k: 3 and Score Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.06it/s]

Generated embeddings with shape (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Copyright © 1995-2018 Clifford Neuman - UNIVERSITY OF SOUTHERN CALIFORNIA - INFORMATION SCIENCES INSTITUTE 
Policy v. Mechanism
• Security policy defines what is and is not allowed
– What confidentiality, integrity, 
and availability actually mean
• S

ecurity mechanism is a method or tool for enforcing 
security policy
– Prevention
– Detection
– Reaction
• Among mechanisms are:
– Mechanisms enforce policy.
– Mechanisms may solve intermediate problems.
▪Authentication, Audit
▪Containment

Copyright © 1995-2018 Clifford Neuman - UNIVERSITY OF SOUTHERN CALIFORNIA - INFORMATION SCIENCES INSTITUTE 
Policy v. Mechanism
• Security policy defines what is and is not allowed
– What confidentiality, integrity, 
and availability actually mean
• Security mechanism is a method or tool for enforcing 
security policy
– Prevention
– Detection
– Reaction
• Among mechanisms are:
– Mechanisms enforce policy.
– Mechanisms may solve intermediate problems.
▪Authentication, Audit
▪Containment

Copyright © 1995-2018 Clifford Neuman - UNIVERSITY OF SOUTHERN CALIFORNIA - INFORMATION SCIENCES INSTITUTE 
A more difficult problem
• Unfortunately, security isn’t that easy.  It 
must be better integrated with the 
application.
– At the level at which it must ultim